In [ ]:
!sudo apt-get update -y
!sudo apt-get install -y pciutils

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,183 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,300 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,005 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,043 kB]
Hit:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:12 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:13 https://ppa.launchpadcontent.net/graphics-drivers/ppa

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
ERROR: This version requires zstd for extraction. Please install zstd and try again:
  - Debian/Ubuntu: sudo apt-get install zstd
  - RHEL/CentOS/Fedora: sudo dnf install zstd
  - Arch: sudo pacman -S zstd


In [ ]:
import subprocess
import os
import time

try:
    # Check if ollama is already installed by trying to run a simple command
    subprocess.run(["ollama", "--version"], check=True, capture_output=True, text=True, timeout=10)
    print("Ollama is already installed.")
except (FileNotFoundError, subprocess.CalledProcessError):
    print("Ollama not found or not working correctly. Attempting to install zstd and then ollama...")

    # 1. Install zstd (identified as missing dependency from previous cell's output)
    print("Installing zstd...")
    zstd_install_cmd = "sudo apt-get update -y && sudo apt-get install -y zstd"
    zstd_result = subprocess.run(zstd_install_cmd, shell=True, capture_output=True, text=True)
    if zstd_result.returncode != 0:
        print("Failed to install zstd. Error:", zstd_result.stderr)
        # Optionally exit or raise an error if zstd is critical and fails
        raise RuntimeError("Failed to install zstd")
    print("zstd installed successfully.")

    # 2. Re-run ollama installation
    print("Re-running ollama installation...")
    ollama_install_cmd = "curl -fsSL https://ollama.com/install.sh | sh"
    ollama_result = subprocess.run(ollama_install_cmd, shell=True, capture_output=True, text=True)
    if ollama_result.returncode != 0:
        print("Failed to install ollama. Error:", ollama_result.stderr)
        raise RuntimeError("Failed to install ollama")
    print("Ollama re-installed successfully.")

    # Give some time for PATH to update or system to recognize new executable, if needed.
    time.sleep(2)

    # Verify ollama installation again
    try:
        subprocess.run(["ollama", "--version"], check=True, capture_output=True, text=True, timeout=10)
        print("Ollama verified after re-installation.")
    except (FileNotFoundError, subprocess.CalledProcessError) as e:
        print(f"Error verifying ollama after re-installation: {e}")
        raise RuntimeError("Ollama still not found after re-installation. Please check logs manually.")

# Finally, attempt to serve ollama
print("Starting ollama serve...")
subprocess.Popen(["ollama", "serve"])
print("Ollama serve command issued. The server should be running in the background.")

Ollama not found or not working correctly. Attempting to install zstd and then ollama...
Installing zstd...
zstd installed successfully.
Re-running ollama installation...
Ollama re-installed successfully.
Ollama verified after re-installation.
Starting ollama serve...
Ollama serve command issued. The server should be running in the background.


In [ ]:
!ollama pull llama3

In [ ]:
!ollama run llama3 "Explain AI in simple words"


Error: Post "http://127.0.0.1:11434/api/generate": EOF


In [ ]:
# This cell is modified to serve Ollama robustly, the actual run will be in the next cell
import subprocess
import time

print("Starting ollama serve...")
# Start ollama serve in the background
serve_process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Give it a moment to start up
time.sleep(5)
print("Ollama serve process initiated. Output will appear in the next cell.")

time=2026-06-08T04:30:31.571Z level=INFO source=routes.go:1919 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: LLAMA_ARG_FIT: LLAMA_ARG_FIT_TARGET: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GO_TEMPLATE:true OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_IGPU_ENABLE: OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MAX_TRANSFER_STREAMS:4 OLLAMA_MODELS:/root/.ollama/models OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_ORIGINS:[http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:* http://0.0.0.0 https://0.0.0.0 http://0.0.0.0:* 

In [ ]:
# Now that the server is (hopefully) up, run the command
import time
import subprocess

# Ensure the server has enough time to start
time.sleep(5)

print("Attempting to run ollama command...")
try:
    # Execute the ollama run command
    result = subprocess.run(["ollama", "run", "llama3", "Hello"], capture_output=True, text=True, check=True)
    print(result.stdout)
    if result.stderr:
        print("Stderr:", result.stderr)
except subprocess.CalledProcessError as e:
    print(f"Error running ollama: {e}")
    print("Stdout:", e.stdout)
    print("Stderr:", e.stderr)
except FileNotFoundError:
    print("Ollama command not found. Ensure it's installed and in your PATH.")


Error: could not connect to ollama server, run 'ollama serve' to start it


In [ ]:
model="llama-3.1-8b-instant"

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key="enter api key",#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
    base_url="https://api.groq.com/openai/v1"
)

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "user", "content": "Explain AI in simple terms"}
    ]
)

print(response.choices[0].message.content)

**What is AI?**

Artificial Intelligence, or AI for short, is a type of technology that allows computers to think and learn like humans. It's a way to make machines smarter and more helpful.

**Think of AI like a Magic Brain**

Imagine you have a smart friend who can:

1. **Learn**: Your friend can learn new things, like your favorite food or your hobby.
2. **Understand**: Your friend can understand what you say, even if you make mistakes.
3. **Remember**: Your friend can remember important information, like your birthday or your favorite movie.
4. **Make decisions**: Your friend can make decisions based on what they learned, like suggesting a restaurant to eat at.

**How does AI work?**

AI uses something called "algorithms" to analyze data and make decisions. It's like a recipe book for computers:

1. **Machine learning**: AI learns from data, like what's in your phone's contact list or your favorite songs.
2. **Deep learning**: AI uses more advanced algorithms to analyze data, like 

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key="enter api key", #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
    base_url="https://api.groq.com/openai/v1"
)

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",   # ✅ updated model
    messages=[
        {
            "role": "user",
            "content": "Explain machine learning in simple words"  # ✅ input fixed
        }
    ]
)

print(response.choices[0].message.content)

**What is Machine Learning?**

Machine learning is a way for computers to learn and improve on their own, without being explicitly programmed.

**Think of it like this:**

Imagine you're a child learning to recognize your family members' faces. At first, you might not be able to tell them apart. But as you see their faces more often, you start to notice patterns and features that make them unique. Over time, you become better at recognizing them.

**How does Machine Learning work?**

Machine learning is similar, but instead of a child, it's a computer program that's designed to learn from data.

Here's a simple example:

1. **Training:** You show the computer a bunch of examples of pictures of cats and dogs.
2. **The computer learns:** It looks for patterns and features that make a picture a cat or a dog.
3. **The computer generalizes:** It creates a set of rules that it can use to recognize new pictures of cats and dogs.

**Types of Machine Learning**

There are three main types of ma

In [ ]:
!pip install groq gtts -q

In [ ]:
from groq import Groq
from gtts import gTTS
from IPython.display import Audio, display



api_key = input("Enter your Groq API key: ")
client = Groq(api_key=api_key)





response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "user", "content": "Explain transformers in AI in simple words"}
    ]
)

text = response.choices[0].message.content

print("\n🤖 AI RESPONSE:\n")
print(text)


tts = gTTS(text=text, lang="en")
tts.save("output.mp3")


display(Audio("output.mp3"))

Enter your Groq API key: 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

🤖 AI RESPONSE:

**What are Transformers in AI?**

Transformers are a type of neural network used in artificial intelligence (AI) and deep learning. They're a game-changer in the field of natural language processing (NLP), which is how computers understand and work with text and speech.

**So, what's the problem?**

Traditionally, AI models like Recurrent Neural Networks (RNNs) and Long Short-Term Memory (LSTM) networks were used for NLP tasks. However, they had a major limitation: they assumed the input data would be sequential and ordered in a specific way. This is known as a "sequential" or "order-dependent" problem.

**Introducing Transformers**

Transformers, introduced in 2017 by Vaswani et al., changed the game. They're based on self-attention mechanisms, which allow the model to attend to all parts of the input data simultaneously and weigh their importance. This is unlike traditional sequential m

In [ ]:
#text to image


from google import genai
from IPython.display import display, HTML

# Initialize the client with your working API key
client = genai.Client(api_key="enter api key")#AQ.Ab8RN6LOu6Np2L57JX9-e4hLjQNzw-eMcReLqOn0CKdvU0XSeQ

prompt = """
Write a complete, single block of raw HTML/SVG code that renders a highly detailed,
futuristic cyberpunk bicycle. Give it glowing blue neon rims, a sleek dark metallic frame,
and place it on a subtle dark digital grid background. Return ONLY the raw HTML/SVG code,
no markdown wrappers.
"""

print("Generating vector graphics... please wait a moment...")

try:
    # Use your working Gemini model to generate the vector code
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
    )

    # Clean up any potential markdown formatting from the response string
    clean_html = response.text.replace("```html", "").replace("```", "").strip()

    # Render the vector artwork cleanly inside your notebook layout
    display(HTML(clean_html))

except Exception as e:
    print(f"\nExecution error: {e}")

Generating vector graphics... please wait a moment...


In [ ]:
#text to image using another prompt

from google import genai
from IPython.display import display, HTML

# Initialize the client with your working API key
client = genai.Client(api_key="enter API key")#AQ.Ab8RN6LOu6Np2L57JX9-e4hLjQNzw-eMcReLqOn0CKdvU0XSeQ

prompt = """
Create a simple robot with glowing eyes in SVG format. Only SVG code.
"""

print("Generating vector graphics... please wait a moment...")

try:
    # Use your working Gemini model to generate the vector code
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
    )

    # Clean up any potential markdown formatting from the response string
    clean_html = response.text.replace("```html", "").replace("```", "").strip()

    # Render the vector artwork cleanly inside your notebook layout
    display(HTML(clean_html))

except Exception as e:
    print(f"\nExecution error: {e}")

Generating vector graphics... please wait a moment...


In [ ]:
# Aggressively uninstall conflicting packages and reinstall with compatible versions
!pip uninstall -y transformers torch torchvision torchaudio pillow psutil -q
!pip install torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2 transformers==4.41.0 psutil "pillow<11" -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 9.5 MB/s eta 0:00:00


In [2]:
import time
import psutil
import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

text = "Machine learning is used for phishing detection and cybersecurity."

bert_model_name = "bert-base-uncased"
gpt_model_name = "gpt2"

bert_tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
bert_model = AutoModel.from_pretrained(bert_model_name).to(DEVICE)

gpt_tokenizer = AutoTokenizer.from_pretrained(gpt_model_name)

if gpt_tokenizer.pad_token is None:
    gpt_tokenizer.pad_token = gpt_tokenizer.eos_token

gpt_model = AutoModelForCausalLM.from_pretrained(gpt_model_name).to(DEVICE)

def benchmark(model, tokenizer, text, model_type):
    process = psutil.Process()
    mem_before = process.memory_info().rss / 1024**2

    inputs = tokenizer(text, return_tensors="pt", truncation=True)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    start = time.time()

    with torch.no_grad():
        if model_type == "BERT":
            model(**inputs)
        else:
            model.generate(inputs["input_ids"], max_new_tokens=30)

    end = time.time()

    mem_after = process.memory_info().rss / 1024**2

    tokens = inputs["input_ids"].shape[1]

    return {
        "Latency(sec)": round(end - start, 4),
        "Tokens": tokens,
        "Memory(MB)": round(mem_after - mem_before, 2),
        "Tokens/sec": round(tokens / (end - start), 2)
    }

bert_result = benchmark(bert_model, bert_tokenizer, text, "BERT")
gpt_result = benchmark(gpt_model, gpt_tokenizer, text, "GPT")

print("BERT:", bert_result)
print("GPT:", gpt_result)
print("Device:", DEVICE)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


BERT: {'Latency(sec)': 0.5257, 'Tokens': 16, 'Memory(MB)': 182.25, 'Tokens/sec': 30.44}
GPT: {'Latency(sec)': 0.7188, 'Tokens': 11, 'Memory(MB)': 195.57, 'Tokens/sec': 15.3}
Device: cuda


In [3]:
import time
import psutil
import pandas as pd
from google import genai

gemini_client = genai.Client(api_key="api key")

# Uncomment the following lines to list available models
print("\nAvailable Models:")
for m in gemini_client.models.list():
    print(f"  Model: {m.name}") # Removed unsupported attribute 'supported_generation_methods'

PROMPTS = [
    "Explain phishing attacks",
    "Write python code for bubble sort",
    "Explain transformers in AI"
]

def benchmark_gemini(prompt):
    process = psutil.Process()
    mem_before = process.memory_info().rss / 1024**2

    start = time.time()

    response = gemini_client.models.generate_content(
        model="gemini-pro-latest", # Changed to a generally available model
        contents=prompt
    )

    end = time.time()

    mem_after = process.memory_info().rss / 1024**2

    output = response.text

    return {
        "model": "Gemini",
        "latency_sec": round(end - start, 3),
        "memory_mb": round(mem_after - mem_before, 2),
        "output_chars": len(output),
        "response": output[:150]
    }

results = []

for p in PROMPTS:
    print("Testing:", p)
    results.append(benchmark_gemini(p))

df = pd.DataFrame(results)

print("\n===== RESULTS =====\n")
print(df)

df.to_csv("benchmark_results.csv", index=False)

print("\nSaved benchmark_results.csv")


Available Models:
  Model: models/gemini-2.5-flash
  Model: models/gemini-2.5-pro
  Model: models/gemini-2.0-flash
  Model: models/gemini-2.0-flash-001
  Model: models/gemini-2.0-flash-lite-001
  Model: models/gemini-2.0-flash-lite
  Model: models/gemini-2.5-flash-preview-tts
  Model: models/gemini-2.5-pro-preview-tts
  Model: models/gemma-4-26b-a4b-it
  Model: models/gemma-4-31b-it
  Model: models/gemini-flash-latest
  Model: models/gemini-flash-lite-latest
  Model: models/gemini-pro-latest
  Model: models/gemini-2.5-flash-lite
  Model: models/gemini-2.5-flash-image
  Model: models/gemini-3-pro-preview
  Model: models/gemini-3-flash-preview
  Model: models/gemini-3.1-pro-preview
  Model: models/gemini-3.1-pro-preview-customtools
  Model: models/gemini-3.1-flash-lite-preview
  Model: models/gemini-3.1-flash-lite
  Model: models/gemini-3-pro-image-preview
  Model: models/gemini-3-pro-image
  Model: models/nano-banana-pro-preview
  Model: models/gemini-3.1-flash-image-preview
  Model: m

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-3.1-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-3.1-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-3.1-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-3.1-pro\nPlease retry in 3.179613483s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [

In [ ]:
from google import genai

client = genai.Client(api_key="enter api key")

for m in client.models.list():
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-pr